# STAGE 13B — HISTORICAL CASE RECOVERY AND TEMPORAL ELIGIBILITY AUDIT
**Malaysia-only FYP | Audit only — no model retraining or data overwriting**

Objectives:
1. Reconcile 287 reviewed → 200 coordinate eligible → 194 static → 121 full temporal.
2. Determine which *identified* static cases are absent in rainfall, rainfall+soil moisture, land cover and full temporal subsets.
3. Audit why recoverable records were excluded (feature missingness, identifier mismatches, date availability, and group construction).
4. Prepare review tables for manual recovery; **never fabricate coordinates, event dates, rainfall, land cover or controls**.

Only propose recoveries here. If a dataset is revised, re-run the **same three-algorithm nested spatial validation**, then reselect and retrain production pipelines. Do not alter Stage 08–12 outputs in this notebook.

# PART A — PATHS AND INPUT VALIDATION
## 1. Libraries and file paths

In [1]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 100)

PROJECT_ROOT = Path(r"C:\Landslide_FYP")
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

INVENTORY_DIR = DEV_DIR / "02_cleaned_inventory"
STATIC_DIR = DEV_DIR / "03_derived_features" / "static_features_final"
TEMPORAL_DIR = DEV_DIR / "03_derived_features" / "temporal_features_final"
MODEL_READY_DIR = DEV_DIR / "04_model_ready"
STAGE04_DIR = DEV_DIR / "01_preprocessing" / "stage04_revised_outputs"
STAGE12_DIR = DEV_DIR / "05_model_results" / "stage12_production_models"

OUTPUT_DIR = DEV_DIR / "05_model_results" / "stage13b_record_recovery_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATHS = {
    "reviewed": INVENTORY_DIR / "malaysia_positive_inventory_final.csv",
    "eligible": INVENTORY_DIR / "malaysia_positive_inventory_ml_eligible.csv",
    "stage04": STAGE04_DIR / "positive_inventory_with_ml_eligibility.csv",
    "static_master": STATIC_DIR / "static_feature_samples_final.csv",
    "static_model": MODEL_READY_DIR / "static_model_1to2.csv",
    "temporal_annotated": TEMPORAL_DIR / "temporal_feature_samples_final_annotated.csv",
    "rainfall": TEMPORAL_DIR / "temporal_model_rainfall.csv",
    "rainfall_soilmoisture": TEMPORAL_DIR / "temporal_model_rainfall_soilmoisture.csv",
    "landcover": TEMPORAL_DIR / "temporal_model_landcover.csv",
    "full_enhanced": TEMPORAL_DIR / "temporal_model_full_enhanced.csv",
    "static_schema": STAGE12_DIR / "static_feature_schema.json",
    "temporal_schema": STAGE12_DIR / "temporal_feature_schema.json",
}

path_audit = pd.DataFrame([
    {"source": name, "path": str(path), "exists": path.exists()}
    for name, path in INPUT_PATHS.items()
])
path_audit.to_csv(OUTPUT_DIR / "stage13b_input_paths.csv", index=False)
display(path_audit)

required = ["static_model", "static_master", "rainfall", "full_enhanced", "temporal_annotated"]
missing = [name for name in required if not INPUT_PATHS[name].exists()]
if missing:
    raise FileNotFoundError(f"Required files missing: {missing}. Check PROJECT_ROOT and Stage 05B/06 exports.")


,source,path,exists
0,reviewed,C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_fina...,True
1,eligible,C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_ml_e...,True
2,stage04,C:\Landslide_FYP\Landslide_FYP_Development\01_preprocessing\stage04_revised_outputs\positive_inv...,True
3,static_master,C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features_final\static_feat...,True
4,static_model,C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\static_model_1to2.csv,True
5,temporal_annotated,C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_...,True
6,rainfall,C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_...,True
7,rainfall_soilmoisture,C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_...,True
8,landcover,C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_...,True
9,full_enhanced,C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_...,True


## 2. Load datasets and standardize identifiers for comparison

In [2]:
tables = {
    name: pd.read_csv(path, low_memory=False)
    for name, path in INPUT_PATHS.items()
    if path.exists() and path.suffix.lower() == ".csv"
}

with open(INPUT_PATHS["static_schema"], encoding="utf-8") as file:
    static_schema = json.load(file)
with open(INPUT_PATHS["temporal_schema"], encoding="utf-8") as file:
    temporal_schema = json.load(file)


def positive_rows(data):
    if "temporal_role" in data.columns:
        return data.loc[data["temporal_role"].astype(str).str.lower().eq("case")].copy()
    if "target" in data.columns:
        return data.loc[pd.to_numeric(data["target"], errors="coerce").eq(1)].copy()
    if "sample_class" in data.columns:
        return data.loc[data["sample_class"].astype(str).str.lower().eq("landslide")].copy()
    return data.copy()


cases = {name: positive_rows(data) for name, data in tables.items()}

counts = pd.DataFrame([
    {"dataset": name, "all_rows": len(data), "case_rows": len(cases[name]),
     "columns": len(data.columns)}
    for name, data in tables.items()
])
counts.to_csv(OUTPUT_DIR / "stage13b_input_counts.csv", index=False)
display(counts)

EXPECTED = {"reviewed":287,"eligible":200,"static_model":194,"rainfall":189,
            "rainfall_soilmoisture":158,"landcover":148,"full_enhanced":121}
for name, expected in EXPECTED.items():
    if name in cases and len(cases[name]) != expected:
        print(f"REVIEW: {name}: expected {expected}, observed {len(cases[name])}")


,dataset,all_rows,case_rows,columns
0,reviewed,287,287,19
1,eligible,200,200,29
2,stage04,287,287,29
3,static_master,3200,200,49
4,static_model,582,194,53
5,temporal_annotated,567,189,92
6,rainfall,567,189,90
7,rainfall_soilmoisture,474,158,91
8,landcover,444,148,91
9,full_enhanced,363,121,92


# PART B — TRUSTED CASE ID CROSSWALK
## 3. Confirm shared event identifiers (never rely on coordinates alone)

In [3]:
# From Stage 13A, sample_id appeared to overlap between static and temporal case rows.
# Inspect the actual values and verify locations/dates before treating them as the same events.
CASE_ID = "sample_id"

CASE_DATASETS = ["static_model", "rainfall", "rainfall_soilmoisture", "landcover", "full_enhanced"]


def normalize_key(series):
    values = series.astype("string").str.strip()
    values = values.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})
    return values


key_rows = []
for name in CASE_DATASETS:
    df = cases[name]
    if CASE_ID not in df.columns:
        raise KeyError(f"{name}: '{CASE_ID}' missing. Review actual identifier schema before joining records.")
    key = normalize_key(df[CASE_ID])
    key_rows.append({"dataset": name,"cases":len(df),"null_ids":int(key.isna().sum()),
                     "duplicate_ids":int(key.dropna().duplicated().sum()),
                     "unique_ids":int(key.nunique(dropna=True))})
    if key.isna().any() or key.dropna().duplicated().any():
        raise ValueError(f"{name}: ID missing or duplicated. Resolve event IDs; do not drop duplicates automatically.")
    cases[name] = df.assign(_case_key=key)

id_audit = pd.DataFrame(key_rows)
id_audit.to_csv(OUTPUT_DIR / "case_key_uniqueness.csv", index=False)
display(id_audit)


,dataset,cases,null_ids,duplicate_ids,unique_ids
0,static_model,194,0,0,194
1,rainfall,189,0,0,189
2,rainfall_soilmoisture,158,0,0,158
3,landcover,148,0,0,148
4,full_enhanced,121,0,0,121


## 4. Audit matching locations and event dates where comparable columns are available

In [4]:
def choose_column(data, candidates):
    lower = {column.lower(): column for column in data.columns}
    for column in candidates:
        if column.lower() in lower:
            return lower[column.lower()]
    return None


LAT_CANDIDATES = ["latitude", "lat", "event_latitude"]
LON_CANDIDATES = ["longitude", "lon", "long", "event_longitude"]
DATE_CANDIDATES = ["event_date", "landslide_date", "date", "event_date_utc", "landslide_event_date"]


def case_profile(data, prefix):
    columns = {"_case_key": "_case_key"}
    lat = choose_column(data, LAT_CANDIDATES)
    lon = choose_column(data, LON_CANDIDATES)
    date = choose_column(data, DATE_CANDIDATES)
    result = data[["_case_key"]].copy()
    result[prefix + "_lat"] = pd.to_numeric(data[lat], errors="coerce") if lat else np.nan
    result[prefix + "_lon"] = pd.to_numeric(data[lon], errors="coerce") if lon else np.nan
    result[prefix + "_date"] = (pd.to_datetime(data[date], errors="coerce", utc=True).dt.date.astype("string")
                                if date else pd.Series(pd.NA, index=data.index, dtype="string"))
    return result, {"dataset":prefix,"latitude_column":lat,"longitude_column":lon,"date_column":date}


static_info, static_cols = case_profile(cases["static_model"], "static")
full_info, full_cols = case_profile(cases["full_enhanced"], "full")
matched_info = full_info.merge(static_info, on="_case_key", how="left", validate="one_to_one", indicator=True)
matched_info["coordinate_difference_degrees"] = np.sqrt(
    (matched_info["static_lat"] - matched_info["full_lat"]) ** 2
    + (matched_info["static_lon"] - matched_info["full_lon"]) ** 2
)
matched_info["date_disagreement"] = (
    matched_info["static_date"].notna()
    & matched_info["full_date"].notna()
    & matched_info["static_date"].ne(matched_info["full_date"])
)
matched_info["coordinate_review"] = matched_info["coordinate_difference_degrees"] > 0.0001
matched_info["identity_review"] = (
    matched_info["_merge"].ne("both")
    | matched_info["coordinate_review"]
    | matched_info["date_disagreement"]
)

matched_info.to_csv(OUTPUT_DIR / "full_temporal_case_identity_verification.csv", index=False)
pd.DataFrame([static_cols, full_cols]).to_csv(OUTPUT_DIR / "event_identity_column_coverage.csv", index=False)
print("Matched full-temporal cases:", int(matched_info["_merge"].eq("both").sum()))
print("Flagged for event-identity review:", int(matched_info["identity_review"].sum()))
print("Important: absent date or coordinate columns mean identity cannot be fully verified automatically.")
display(matched_info.loc[matched_info["identity_review"]].head(30))


Matched full-temporal cases: 121
Flagged for event-identity review: 0
Important: absent date or coordinate columns mean identity cannot be fully verified automatically.


,_case_key,full_lat,full_lon,full_date,static_lat,static_lon,static_date,_merge,coordinate_difference_degrees,date_disagreement,coordinate_review,identity_review


# PART C — TRACK EXCLUDED CASES THROUGH TEMPORAL SUBSETS
## 5. Build per-case retention and exclusion table

In [5]:
static_cases = cases["static_model"].copy()
retention = static_cases[["_case_key"]].copy()

for name in ["rainfall", "rainfall_soilmoisture", "landcover", "full_enhanced"]:
    valid_keys = set(cases[name]["_case_key"].dropna())
    retention["in_" + name] = retention["_case_key"].isin(valid_keys)

retention["first_absent_stage"] = np.select(
    [~retention["in_rainfall"],
     retention["in_rainfall"] & ~retention["in_rainfall_soilmoisture"],
     retention["in_rainfall"] & ~retention["in_landcover"],
     retention["in_rainfall_soilmoisture"] & retention["in_landcover"] & ~retention["in_full_enhanced"]],
    ["not_in_rainfall", "not_in_rainfall_soilmoisture", "not_in_landcover", "not_in_full_despite_component_eligibility"],
    default="retained_full_or_check_non_nested_membership"
)

# Rainfall+moisture and land-cover are independent/partly overlapping subsets.
# Therefore, first_absent_stage is a descriptive precedence label, NOT necessarily causal.
for name, frame in cases.items():
    if name in ["rainfall", "rainfall_soilmoisture", "landcover", "full_enhanced"]:
        outside = set(frame["_case_key"]) - set(static_cases["_case_key"])
        if outside:
            print(f"WARNING: {name} has {len(outside)} case IDs outside static training positives. Review the ID semantics.")

retention.to_csv(OUTPUT_DIR / "case_level_retention_matrix.csv", index=False)
summary = pd.DataFrame([
    {"subset":name,"static_cases":len(retention),"retained_static_cases":int(retention["in_"+name].sum()),
     "absent_static_cases":int((~retention["in_"+name]).sum())}
    for name in ["rainfall", "rainfall_soilmoisture", "landcover", "full_enhanced"]
])
summary.to_csv(OUTPUT_DIR / "retention_overlap_summary.csv", index=False)
display(summary)
display(retention["first_absent_stage"].value_counts().rename_axis("category").reset_index(name="cases"))


,subset,static_cases,retained_static_cases,absent_static_cases
0,rainfall,194,189,5
1,rainfall_soilmoisture,194,158,36
2,landcover,194,148,46
3,full_enhanced,194,121,73


,category,cases
0,retained_full_or_check_non_nested_membership,121
1,not_in_landcover,37
2,not_in_rainfall_soilmoisture,31
3,not_in_rainfall,5


## 6. Diagnose feature gaps in the largest available annotated temporal case table

In [6]:
annotated = positive_rows(tables["temporal_annotated"]).copy()
if CASE_ID not in annotated.columns:
    raise KeyError("Annotated temporal case table lacks sample_id. Inspect Stage 05B construction and its identifier fields.")
annotated["_case_key"] = normalize_key(annotated[CASE_ID])
if annotated["_case_key"].dropna().duplicated().any():
    print("WARNING: duplicated annotated sample_id; diagnostic join is blocked to avoid multiplying events.")
    annotated_unique = pd.DataFrame(columns=["_case_key"])
else:
    annotated_unique = annotated

REQUIRED_TEMPORAL = [
    "rain_1d_mm", "rain_3d_mm", "rain_7d_mm", "rain_30d_mm",
    "soil_moisture_0_7cm_7d_mean", "soil_moisture_7_28cm_7d_mean",
    "landcover_harmonised", "distance_waterway_current_m"
]

source_detail = annotated_unique[["_case_key"]].copy()
for feature in REQUIRED_TEMPORAL:
    source_detail["missing_" + feature] = (
        annotated_unique[feature].isna() if feature in annotated_unique.columns
        else pd.Series(True, index=annotated_unique.index)
    )

observed = retention.merge(source_detail, on="_case_key", how="left", validate="one_to_one", indicator="annotated_join")
missing_flags = ["missing_" + feature for feature in REQUIRED_TEMPORAL]
observed["missing_temporal_feature_count"] = observed[missing_flags].fillna(True).sum(axis=1)
observed["annotated_record_missing"] = observed["annotated_join"].eq("left_only")
observed["potentially_recoverable_from_existing_columns"] = (
    (~observed["in_full_enhanced"])
    & (~observed["annotated_record_missing"])
    & observed["missing_temporal_feature_count"].eq(0)
)

observed.to_csv(OUTPUT_DIR / "temporal_case_missingness_and_recovery_candidates.csv", index=False)

feature_gap_rows = []
for feature in REQUIRED_TEMPORAL:
    flag = "missing_" + feature
    feature_gap_rows.append({
        "feature": feature,
        "missing_among_static_not_full": int(observed.loc[~observed["in_full_enhanced"], flag].fillna(True).sum()),
        "missing_among_full": int(observed.loc[observed["in_full_enhanced"], flag].fillna(True).sum()),
        "source_column_exists": feature in annotated.columns,
    })
feature_gap = pd.DataFrame(feature_gap_rows)
feature_gap.to_csv(OUTPUT_DIR / "missingness_reason_breakdown.csv", index=False)
display(feature_gap)
print("Not in full temporal:", int((~observed["in_full_enhanced"]).sum()))
print("Present in annotated with all listed temporal features:", int(observed["potentially_recoverable_from_existing_columns"].sum()))
print("These are REVIEW CANDIDATES, not verified recoveries. Check dates, sampling, static predictors and Stage 05B filtering.")


,feature,missing_among_static_not_full,missing_among_full,source_column_exists
0,rain_1d_mm,5,0,True
1,rain_3d_mm,5,0,True
2,rain_7d_mm,5,0,True
3,rain_30d_mm,5,0,True
4,soil_moisture_0_7cm_7d_mean,22,0,True
5,soil_moisture_7_28cm_7d_mean,22,0,True
6,landcover_harmonised,20,0,True
7,distance_waterway_current_m,5,0,True


Not in full temporal: 73
Present in annotated with all listed temporal features: 36
These are REVIEW CANDIDATES, not verified recoveries. Check dates, sampling, static predictors and Stage 05B filtering.


## 7. Check event-date availability, without fabricating dates

In [7]:
date_candidates = ["event_date", "landslide_date", "date", "event_date_utc", "landslide_event_date"]
raw_date_column = choose_column(annotated, date_candidates)

if raw_date_column:
    dates = pd.to_datetime(annotated[raw_date_column], errors="coerce", utc=True)
    date_audit = pd.DataFrame({"_case_key": annotated["_case_key"],
                               "event_date_raw": annotated[raw_date_column],
                               "event_date_parseable": dates.notna(),
                               "event_year": dates.dt.year})
else:
    date_audit = pd.DataFrame({"_case_key": annotated["_case_key"],
                               "event_date_raw": pd.NA,
                               "event_date_parseable": False,
                               "event_year": np.nan})
    print("No recognized date column. Review Stage 05B date schema manually.")

case_dates = retention.merge(date_audit, on="_case_key", how="left", validate="one_to_one")
case_dates.to_csv(OUTPUT_DIR / "temporal_event_date_audit.csv", index=False)
print("Cases with a parseable annotated event date:", int(case_dates["event_date_parseable"].fillna(False).sum()))
print("Note: date parsing alone does not establish that rainfall/soil moisture were extracted strictly before the event.")


No recognized date column. Review Stage 05B date schema manually.
Cases with a parseable annotated event date: 0
Note: date parsing alone does not establish that rainfall/soil moisture were extracted strictly before the event.


# PART D — STATIC CASES REMOVED AT STATIC QC
## 8. Identify the six missing cases and review source columns

In [8]:
master = positive_rows(tables["static_master"]).copy()
model = cases["static_model"].copy()

id_candidates = ["sample_id", "positive_id"]
master_key = next((col for col in id_candidates if col in master.columns), None)
model_key = next((col for col in id_candidates if col in model.columns), None)

if master_key and model_key:
    master["_candidate_key"] = normalize_key(master[master_key])
    model["_candidate_key"] = normalize_key(model[model_key])
    if master["_candidate_key"].dropna().duplicated().any():
        print("REVIEW: master sample IDs not unique; cannot safely identify the six excluded cases by ID.")
        static_excluded = pd.DataFrame(columns=["_candidate_key"])
    else:
        static_excluded = master.loc[~master["_candidate_key"].isin(set(model["_candidate_key"]))].copy()
else:
    print("REVIEW: no shared sample ID column. Do not identify excluded rows by coordinate alone.")
    static_excluded = pd.DataFrame(columns=["_candidate_key"])

static_excluded.to_csv(OUTPUT_DIR / "static_qc_excluded_cases_review.csv", index=False)
print("Static master positives:", len(master))
print("Static training positives:", len(model))
print("ID-based master exclusions (review semantics):", len(static_excluded))
print("If count is not six, inspect Stage 05A soil QC and the sample-id crosswalk.")
if len(static_excluded):
    display(static_excluded.head(12))


Static master positives: 200
Static training positives: 194
ID-based master exclusions (review semantics): 6
If count is not six, inspect Stage 05A soil QC and the sample-id crosswalk.


,sample_id,sample_class,target,longitude,latitude,elevation_m,slope_deg,aspect_deg,terrain_valid,soil_clay,soil_sand,soil_silt,soil_bulk_density,soil_complete,distance_major_fault_m,fault_distance_available,core_static_valid,all_stage05A_features_available,soil_clay_is_zero,soil_sand_is_zero,soil_silt_is_zero,soil_bulk_density_is_zero,soil_zero_count,soil_all_zero,soil_complete_qc,soil_valid_qc,lithology_polygon_coverage_count,lithology_spatially_covered,national_core_static_valid,geology_enhanced_valid,soil_missing_any,soil_needs_recovery,soil_recovery_attempted,soil_recovery_success,soil_recovery_max_pixel_distance,soil_final_all_zero,soil_final_missing_any,soil_valid_final,geology_polygon_count,geology_age,geology_age_code,geology_unit_name,geology_unit_code,geology_unit_type,geology_source_layers,geology_spatially_covered,geology_attribute_available,national_core_static_valid_final,geology_enhanced_valid_final,_candidate_key
49,POS_0069,landslide,1.0,101.70400,3.149800,80,15,353,True,0,0,0,0,True,9879.250446,True,True,True,True,True,True,True,4,True,True,False,2,True,False,False,False,True,True,False,NaN,True,False,False,2,Caroboniferous - Permian,5500.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Kuala Lumpur | Rock age of Kuala Lumpur,True,True,False,False,POS_0069
66,POS_0093,landslide,1.0,101.41690,3.008900,6,2,0,True,0,0,0,0,True,35826.993164,True,True,True,True,True,True,True,4,True,True,False,2,True,False,False,False,True,True,False,NaN,True,False,False,2,Quaternary,1000.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Selangor | Rock age of Selangor,True,True,False,False,POS_0093
79,POS_0118,landslide,1.0,100.31801,5.395580,6,1,315,True,0,0,0,0,True,59517.234871,True,True,True,True,True,True,True,4,True,True,False,2,True,False,False,False,True,True,False,NaN,True,False,False,2,Quaternary,1000.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Pulau_Pinang | Rock age of Pulau_Pinang,True,True,False,False,POS_0118
104,POS_0163,landslide,1.0,101.70360,3.149700,85,14,305,True,0,0,0,0,True,9903.926558,True,True,True,True,True,True,True,4,True,True,False,2,True,False,False,False,True,True,False,NaN,True,False,False,2,Caroboniferous - Permian,5500.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Kuala Lumpur | Rock age of Kuala Lumpur,True,True,False,False,POS_0163
144,POS_0229,landslide,1.0,101.60210,3.019400,16,4,135,True,0,0,0,0,True,27260.441584,True,True,True,True,True,True,True,4,True,True,False,2,True,False,False,False,True,True,False,NaN,True,False,False,2,Ordovician-Silurian,8000.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Selangor | Rock age of Selangor,True,True,False,False,POS_0229
158,POS_0246,landslide,1.0,101.73814,3.166872,41,5,270,True,0,0,0,0,True,6870.387788,True,True,True,True,True,True,True,4,True,True,False,2,True,False,False,False,True,True,False,NaN,True,False,False,2,Ordovician-Silurian,8000.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Selangor | Rock age of Selangor,True,True,False,False,POS_0246


# PART E — DECISION AND NEXT STEPS
## 9. Preserve performance metrics and define recovery gates

In [9]:
CHECKS = [
    {"gate":"Static vs temporal case IDs", "status": "REVIEW REQUIRED", "detail":"Verify same event ID, location and event date where available"},
    {"gate":"Temporal missingness", "status": "AUDITED", "detail":"Check per-case missing feature flags and source availability"},
    {"gate":"Event timestamps", "status": "REVIEW REQUIRED", "detail":"Confirm source observation windows are strictly before event/reference date"},
    {"gate":"Static six-case QC", "status": "REVIEW REQUIRED", "detail":"Validate soil nodata and sample ID semantics before including"},
    {"gate":"Historical control construction", "status": "REVIEW REQUIRED", "detail":"Any recovered positive needs defensible temporally matched controls"},
    {"gate":"LR/RF/XGBoost benchmark", "status": "PRESERVED", "detail":"Original Stage 08–11 nested spatial metrics remain valid for original sample set"},
    {"gate":"National mapping", "status": "NOT VERIFIED", "detail":"Separate Stage 13C nationwide predictor raster coverage audit required"},
]
gates = pd.DataFrame(CHECKS)
gates.to_csv(OUTPUT_DIR / "stage13b_decision_gate.csv", index=False)

print("STAGE 13B AUDIT OUTPUT:", OUTPUT_DIR)
print("No historical records were added, deleted, or modified.")
print("No models were retrained. Stage 08–12 benchmark and production artifacts remain unchanged.")
display(gates)


STAGE 13B AUDIT OUTPUT: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b_record_recovery_audit
No historical records were added, deleted, or modified.
No models were retrained. Stage 08–12 benchmark and production artifacts remain unchanged.


,gate,status,detail
0,Static vs temporal case IDs,REVIEW REQUIRED,"Verify same event ID, location and event date where available"
1,Temporal missingness,AUDITED,Check per-case missing feature flags and source availability
2,Event timestamps,REVIEW REQUIRED,Confirm source observation windows are strictly before event/reference date
3,Static six-case QC,REVIEW REQUIRED,Validate soil nodata and sample ID semantics before including
4,Historical control construction,REVIEW REQUIRED,Any recovered positive needs defensible temporally matched controls
5,LR/RF/XGBoost benchmark,PRESERVED,Original Stage 08–11 nested spatial metrics remain valid for original sample set
6,National mapping,NOT VERIFIED,Separate Stage 13C nationwide predictor raster coverage audit required


## Outputs to send for review

- `case_key_uniqueness.csv`
- `full_temporal_case_identity_verification.csv`
- `event_identity_column_coverage.csv`
- `case_level_retention_matrix.csv`
- `retention_overlap_summary.csv`
- `temporal_case_missingness_and_recovery_candidates.csv`
- `missingness_reason_breakdown.csv`
- `temporal_event_date_audit.csv`
- `static_qc_excluded_cases_review.csv`
- `stage13b_decision_gate.csv`

**Privacy / data integrity note:** Keep original coordinates/event metadata available for verification. Do not infer a specific event from duplicate coordinates. A complete national mapping gate still requires Stage 13C.